In [1]:
import json
import re
import ntpath
import os

In [2]:
def load_text(path_to_json):
    with open(path_to_json) as f:
        dictionary = json.load(f)
        if 'text' in dictionary:
            text = dictionary['text']
        else:
            return 'no text found'
    text = re.sub(r'BÜND(-\n)?NIS((-\n)?SES)?\s*?90\/\s*(DIE\s*?)?GRÜ(-\n)?NEN?','GRÜNE', text, flags=re.IGNORECASE) # That's a long short form for a party...
    text = re.sub(r'\n-\n', '-', text) # weird formatting probably due to digitalization from printed mediums
    text = re.sub(r'DIE\s*?LINKE|PDS/\s*?Linke\s*?Liste|PDS/LL', 'DIE LINKE', text)
    text = re.sub(r'F\.D\.P\.', 'FDP', text)
    return text

In [3]:
datapath = r'data\protokolle\20_012_2022-01-14.json'
text = load_text(datapath)

In [4]:
name_match = ( '('
              +  r'(?:Dr\.\s)?'   # Optional "Dr. "
              + r'(?:\w+(?:-\w+)?\s)'   # First name (or hyphenated first name)
              + r'(?:\w+\.\s)?'   # Optional middle initial
              + r'\w+(?:-\w+)?'   # Last name (or hyphenated last name)
              + ')'
             )
# Dr. Marie-Agnes Strack-Zimmermann | Dr. Harald L. Töpfer

parties = ['CDU/CSU', 'GRÜNE','SPD', 'FDP', 'AfD', 'DIE LINKE', 'KPD', 'BP', 'DP', 'WAV', 'Z', 'fraktionslos', ] # Die Abkürzungen der wichtigsten - auch historischen - Parteien
partei_match = r'(CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)'

kommentar_match = re.compile(rf'{name_match}\s+' # comments are comprised of a name
                                rf'\[{partei_match}\]:\s' # ...followed by the party in square brackets
                                r'([\s\S]*?)' # ...followed by the comment text including newlines
                                r'[-–—\)]', # ...ended by some sort of hyphen or closing round brackets
                                flags = re.UNICODE)

beifall_match = re.compile(r'[-–—\s\(]' # Applause is comprised of a leading hyphen or opening parenthesis
                           'Beifall'    # ...followed by "Beifall"
                           r'[\s\S]*?'  # ...followed by a sentence including the party names
                           r'[-–—\)]',  # ...ended by some sort of hyphen or closing round parenthesis
                           flags = re.UNICODE)

zuruf_match = re.compile(r'[-–—\s\(]'   # Comments with unknown speakers but known party affiliation are comprised of a leading hyphen or opening parenthesis
                         r'Zuruf .*?'   # ... followed by 'Zuruf' and some more irrelevant fillwords
                         rf'{partei_match}:\s'  # ... folllowed by the party name and :
                         r'([\s\S]*?)'     # ... followed by the text of the comment
                         r'[-–—\)]',    # ... ended by some sort of hyphen or closing round parenthesis
                        flags = re.UNICODE)


speaker_match = re.compile(
    rf'{name_match}\s(?:\(\w+\)\s)?\({partei_match}\):'  # Matches speaker's name followed by party name in parentheses, e.g., "Harald Töpfer (Party):"
    '|'
    r'\n'  # Newline
    rf'{name_match}, ([\w\s]*?)(?:Saatssekretär|Staatsminister|Bundesminister|Bundeskanzler)[\w\s]*?\n?'  # Matches name followed by official title and fillwords with no more than one newline, e.g. "Harald Töpfer, Bundeskanzler" "Ronald Wiesel, Staatssekretär beim Bundeskanzler"
    r'[\w\s]*?:',  # Matches any remaining text up to :
    flags=re.UNICODE
)
# bottom part of the regex (after |) has two capture-groups too so that the alignment of names is not out of order when splitting
# conveniently, no distinction between male and female Staatsminister(in) , Saatssekretär(in), Bundeskanzler(in) etc. has to be made as they start with the same chars

In [5]:
speaker_match

re.compile(r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)(?:\w+\.\s)?\w+(?:-\w+)?)\s(?:\(\w+\)\s)?\((CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE|fraktionslos)\):|\n((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)(?:\w+\.\s)?\w+(?:-\w+)?), ([\w\s]*?)(?:Saatssekretär|Staatsminister|Bundesminister|Bundeskanzler)[\w\s]*?\n?[\w\s]*?:',
           re.UNICODE)

In [6]:
print(re.findall(name_match, 'Dr. Harald L. Töpfer'))
print(re.findall(speaker_match, 'Dr. Marie-Agnes Strack-Zimmermann (FDP): '))

['Dr. Harald L. Töpfer']
[('Dr. Marie-Agnes Strack-Zimmermann', 'FDP', '', '')]


### Get List of all Representatives and their party affiliation since 1949
see Notebook get_abgeordneten_data for more

In [7]:
import csv
bt_tuples = []
with open('./data/abgeordnete.csv', 'r', newline= '',encoding='utf8') as file:
    reader = csv.reader(file)
    next(reader) # skip first row
    for row in reader:
        bt_tuples.append(tuple(row))


In [8]:
def find_party_by_name(name, tuple_list):
    tuple_list_sorted = sorted(tuple_list, key = lambda x: x[2], reverse = True) # we focus on 
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list_sorted:
        if name in item[0]:
            if item[1] != 'CSU' and item[1] != 'CDU':
                return item[1]
            else:
                return 'CDU/CSU'  
    return "<unknown>"

print(find_party_by_name('Dr. Robert Habeck', bt_tuples))

GRÜNE


the sort helps in avoiding errors. We (for now) mostly look at new data.  
This is why handling the few name conflicts present in the list by just taking the newer one is sufficient enough

# Get information out of protocol

In [12]:
def parse_protocol(path_to_json, path_to_output_dir):
    text = load_text(path_to_json)
    if load_text == 'no text found':
        print(f"Processed protocol {ntpath.basename(path_to_json)}. NO TEXT FOUND")
    
    count_zusammengefasst = 0 #debugging

    speeches_raw = re.split(speaker_match, text)[1:]
    speeches = []
    for i in range(0, len(speeches_raw), 5):
        if speeches_raw[i] != None and speeches_raw[i+1] != None:
            name = speeches_raw[i].strip()    
            party = speeches_raw[i+1].strip()
            if party == 'CSU' or party == 'CDU': # sometimes the the secretary tasked with writing everything down forgets to put CDU/CSU instead of CSU or CDU
                party = 'CDU/CSU'
        else:
            name = speeches_raw[i+2].replace('\n', ' ').strip() # sometimes the formatting is all over the place e.g. "Olaf\nScholz" instead of "Olaf Scholz" :)
            party = find_party_by_name(name, bt_tuples)

        speech = {
            'speaker':{
                'name':name,
                'party':party
            },
            'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(\nAnlage)',speeches_raw[i+4])[0].strip() # handle end of File and interruptions by the Bundestagspräsident(in) or Vice Bundestagspräsident(in)
        }


        #debugging
        if party not in parties:
            print(f"    Error at {speech}")
            print(f"    Speaker not found. Either Speaker is not part of BT,  RegEx falsely matched expression or protocol has errors. Speech will not appear in parsed version.")
            continue
            # This is triggered mostly by guest-speakers and name-typos. Also, sometimes a state-secretary has not been member of the BT before becoming state-secretary
        #end_debugging
        
        if len (speeches) >=1 and speeches[-1]['speaker'] == speech['speaker']:
            speeches[-1]['text'] += '\n' + speech['text'] # append interrupted speeches by same speaker e.g. after interruptions by Bundestagspräsident(in)
            count_zusammengefasst +=1
        else:
            speeches.append(speech)

        
        comments = []
        # comments with known speaker
        for match in re.finditer(kommentar_match,speech['text']):
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3).strip(),
                'preceding_context': speech['text'][:match.start()] # include all text until comment for later training of LLM
            }
            comments.append(comment)
        
        # comments with unknown speaker
        for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
            comment = {'commentator':{
                    'name': '<unknown>',
                    'party': match.group(1)
                },
                'text': match.group(2),
                'preceding_context': speech['text'][:match.start()]
            }
            comments.append(comment)

        speech['comments'] = comments

        # applause
        beifall = re.findall(beifall_match, speech['text'])
        beifall = ''.join(beifall)
        beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
        beifall_counts = {party: beifall.count(f' {party}') for party in parties}
        speech['applause'] = beifall_counts
  
    json_string = json.dumps(speeches, indent=4)
    print(f"Processed protocol {ntpath.basename(path_to_json)}. #speeches = {len(speeches)} #speeches_concatenated = {count_zusammengefasst}")
    # Write JSON string to a file
    with open(f"{path_to_output_dir}{ntpath.basename(path_to_json)}", "w") as json_file:
        json_file.write(json_string)


In [10]:
data_path = r'./data/opendata_api/protocols/'
output_path = r'./data/opendata_api/parsed_protocols/'

In [13]:
for filename in os.listdir(data_path):
    parse_protocol(data_path + filename, output_path)

Processed protocol 12_013_1991-03-12.json. #speeches = 72 #speeches_concatenated = 14
Processed protocol 12_014_1991-03-13.json. #speeches = 90 #speeches_concatenated = 28
Processed protocol 12_015_1991-03-14.json. #speeches = 93 #speeches_concatenated = 30
Processed protocol 12_016_1991-03-15.json. #speeches = 26 #speeches_concatenated = 4
Processed protocol 12_017_1991-03-20.json. #speeches = 33 #speeches_concatenated = 20
Processed protocol 12_018_1991-03-21.json. #speeches = 147 #speeches_concatenated = 45
Processed protocol 12_019_1991-03-22.json. #speeches = 59 #speeches_concatenated = 21
Processed protocol 12_020_1991-04-17.json. #speeches = 63 #speeches_concatenated = 18
Processed protocol 12_021_1991-04-18.json. #speeches = 110 #speeches_concatenated = 45
Processed protocol 12_022_1991-04-19.json. #speeches = 27 #speeches_concatenated = 13
Processed protocol 12_023_1991-04-25.json. #speeches = 137 #speeches_concatenated = 57
Processed protocol 12_024_1991-04-26.json. #speeches

# Debugging

In [15]:
data_path = r'data\opendata_api\protocols\13_182_1997-06-13.json'
data_path = r'.\data\opendata_api\protocols\20_169_2024-05-16.json'
data_path = r'data\opendata_api\protocols\12_004_1991-01-18.json'
data_path = r'data\opendata_api\protocols\16_092_2007-03-30.json'
data_path = r'data\opendata_api\protocols\20_114_2023-07-05.json'
output_path = './data/opendata_api/parsed_protocols/'
parse_protocol(data_path, output_path)

    Error at {'speaker': {'name': 'Olaf\nScholz', 'party': '<unknown>'}, 'text': 'Fragestunde\n\n\nDrucksache 20/7518\n\n\n\nMündliche Frage 1\n\nBernd Schattner (AfD)\n\n\nPreisentwicklung von Wohnimmobilien im ländlichen und im städtischen Raum\n\n\nAntwort\n\n\nSören Bartol, Parl. Staatssekretär BMWSB\n\n\n\nZusatzfragen\n\n\nBernd Schattner (AfD)\n\n\n\nLars Rohwer (CDU/CSU)\n\n\n\nDr.\xa0Jan-Marco Luczak (CDU/CSU)\n\n\n\nCaren Lay (DIE LINKE)\n\n\n\nMichael Kießling (CDU/CSU)\n\n\n\nMechthild Heil (CDU/CSU)\n\n\n\n\nMündliche Frage 2\n\nBernd Schattner (AfD)\n\n\nAktuelle Situation im Bauwesen\n\n\nAntwort\n\n\nSören Bartol, Parl. Staatssekretär BMWSB\n\n\n\nZusatzfragen\n\n\nBernd Schattner (AfD)\n\n\n\nLars Rohwer (CDU/CSU)\n\n\n\nMichael Kießling (CDU/CSU)\n\n\n\n\nMündliche Frage 3\n\nStephan Brandner (AfD)\n\n\nUrsachen für den Wohnungsmangel\n\n\nAntwort\n\n\nSören Bartol, Parl. Staatssekretär BMWSB\n\n\n\nZusatzfragen\n\n\nStephan Brandner (AfD)\n\n\n\nPetra Nicolaisen (CDU

In [16]:
text = load_text(data_path)

with open('./data/test/' + ntpath.basename(data_path)[:-4]+ 'txt', 'w', encoding = 'utf8') as f:
        f.write(text)

# Sonderfälle

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml

Sonderfall Einwürfe bei 1951-Protokollen. und 1971 "(Abg. <nachname>: <text>)
1951: Auch neuer Textanfang: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)? :
1971 + 1981: (Dr.)? <nachname> (<partei>) (<Wahlkreis>)?: